# 27 - Build Production Triton Bundle

Gera um pacote utilizável para Linux/NVIDIA/Colab com runtime Triton, manifesto de produção, prewarm de shapes críticos e zip para download.

Objetivos:
1. reconstruir a pasta distribuível sem depender de build antigo;
2. copiar para um bundle de produção dedicado;
3. executar prewarm dos shapes críticos identificados no notebook 26;
4. salvar `production_manifest.json` e `prewarm_profile.json`;
5. gerar zip baixável para uso local em ambiente compatível.


In [ ]:
# Dependências principais.
import os, sys, subprocess, importlib.util, importlib.metadata

required = {
    'torch': 'torch',
    'triton': 'triton',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'peft': 'peft',
    'safetensors': 'safetensors',
    'sentencepiece': 'sentencepiece',
    'psutil': 'psutil',
    'packaging': 'packaging',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from packaging.version import Version
def installed_version(package_name):
    try:
        return Version(importlib.metadata.version(package_name))
    except importlib.metadata.PackageNotFoundError:
        return None

torchao_version = installed_version('torchao')
if torchao_version is not None and torchao_version < Version('0.16.0'):
    print(f'Atualizando torchao incompatível: {torchao_version} -> >=0.16.0')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao>=0.16.0'])

print('Dependências prontas.')

In [ ]:
# Bootstrap Colab/local e paths oficiais.
from pathlib import Path
import json, time, gc, shutil, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
BUILD_DIR = PROJECT_ROOT / 'build'
RUNTIME_DIR = PROJECT_ROOT / 'runtime' / 'packed_recovered_l78'
DIST_DIR = BUILD_DIR / 'distributable_packed_recovered_l78'
DATA_PATH = PROJECT_ROOT / 'data' / 'gemma4_100_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BUILD_DIR.mkdir(parents=True, exist_ok=True)
DIST_DIR.mkdir(parents=True, exist_ok=True)
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUNTIME_DIR =', RUNTIME_DIR)
print('DIST_DIR =', DIST_DIR)

In [ ]:
# Garante que a pasta distribuível contenha o runtime Triton atualizado.
# Diferente do 21/22, este notebook se autocorrige: se o Drive tiver runtime antigo,
# baixa os arquivos versionados do GitHub antes de montar o pacote.
import urllib.request

STUDENT_DIR = CHECKPOINTS_DIR / 'student_pruned'
ADAPTER_DIR = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
PACKED_SOURCE_DIR = next((p for p in [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR] if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)
RUNTIME_RAW_BASE = 'https://raw.githubusercontent.com/cesarfavero/RPT/main/gemma4_theory/runtime/packed_recovered_l78'
RUNTIME_FILES = [
    'README.md',
    'requirements.txt',
    'loader.py',
    'run_inference.py',
    'native_loader.py',
    'run_native_inference.py',
    'triton_loader.py',
    'run_triton_inference.py',
    'package_manifest.template.json',
]

def copytree_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    shutil.copytree(src, dst)

def copy_file_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)

def download_runtime_file(filename):
    url = f'{RUNTIME_RAW_BASE}/{filename}'
    target = RUNTIME_DIR / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    print('baixando runtime ausente:', filename)
    with urllib.request.urlopen(url, timeout=30) as response:
        content = response.read()
    target.write_bytes(content)

def ensure_runtime_templates():
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    missing = [filename for filename in RUNTIME_FILES if not (RUNTIME_DIR / filename).exists()]
    for filename in missing:
        download_runtime_file(filename)
    still_missing = [str(RUNTIME_DIR / filename) for filename in RUNTIME_FILES if not (RUNTIME_DIR / filename).exists()]
    if still_missing:
        raise FileNotFoundError(
            'Runtime continua incompleto após auto-download:\n' + '\n'.join(still_missing)
        )
    return missing

def ensure_distributable():
    if PACKED_SOURCE_DIR is None:
        raise FileNotFoundError('Packed source not found in exports/ternary_mlp_l78 or deploy')
    runtime_downloaded = ensure_runtime_templates()
    required_sources = [STUDENT_DIR, ADAPTER_DIR, RUNTIME_DIR]
    missing = [str(p) for p in required_sources if not Path(p).exists()]
    if missing:
        raise FileNotFoundError('Missing sources:\n' + '\n'.join(missing))

    for src, dst in [(STUDENT_DIR, DIST_DIR / 'student_pruned'), (ADAPTER_DIR, DIST_DIR / 'adapter_recovered_l78'), (RUNTIME_DIR, DIST_DIR / 'runtime')]:
        copytree_clean(src, dst)
    (DIST_DIR / 'packed').mkdir(parents=True, exist_ok=True)
    copy_file_clean(PACKED_SOURCE_DIR / 'manifest.json', DIST_DIR / 'packed' / 'manifest.json')
    copy_file_clean(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin', DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin')
    return runtime_downloaded

runtime_downloaded = ensure_distributable()
runtime_dist = DIST_DIR / 'runtime'
required_runtime_files = [
    runtime_dist / 'triton_loader.py',
    runtime_dist / 'run_triton_inference.py',
    runtime_dist / 'native_loader.py',
]
missing_runtime = [str(p) for p in required_runtime_files if not p.exists()]
if missing_runtime:
    raise FileNotFoundError(
        'Runtime distribuível continua desatualizado. Ausentes:\n' + '\n'.join(missing_runtime)
    )

# Garante prioridade do runtime distribuível e evita módulos antigos cacheados em sessão Colab.
runtime_dist_str = str(runtime_dist)
if runtime_dist_str in sys.path:
    sys.path.remove(runtime_dist_str)
sys.path.insert(0, runtime_dist_str)
for module_name in ['triton_loader', 'native_loader', 'loader']:
    if module_name in sys.modules:
        del sys.modules[module_name]

print('distributable ready:', DIST_DIR)
print('runtime_dist =', runtime_dist)
print('runtime_downloaded =', runtime_downloaded)
print('runtime files:', sorted(p.name for p in runtime_dist.glob('*.py')))


In [ ]:
# Configuração do bundle de produção.
PROD_DIR = BUILD_DIR / 'production_triton_packed_recovered_l78'
PROD_RESULTS_DIR = RESULTS_DIR / 'production_bundle'
PROD_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
MAX_NEW_TOKENS = 12

# Shapes críticos detectados no notebook 26.
PREWARM_QUESTIONS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
    'Qual esporte usa raquete, rede e peteca?',
    'Qual gás as plantas absorvem principalmente para fazer fotossíntese?',
    'Qual comando Linux lista arquivos de um diretório?',
]

PROD_PACKAGE_NAME = 'production_triton_packed_recovered_l78'
print('PROD_DIR =', PROD_DIR)


In [ ]:
import statistics
import psutil
import torch


def human_bytes(n):
    n = float(n)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024.0 or unit == 'TB':
            return f'{n:.2f} {unit}'
        n /= 1024.0


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def snapshot_memory(label):
    rss = psutil.Process().memory_info().rss
    item = {'label': label, 'rss_bytes': rss, 'rss_human': human_bytes(rss)}
    if torch.cuda.is_available():
        item.update({
            'cuda_device': torch.cuda.get_device_name(0),
            'cuda_allocated_bytes': torch.cuda.memory_allocated(),
            'cuda_reserved_bytes': torch.cuda.memory_reserved(),
            'cuda_max_allocated_bytes': torch.cuda.max_memory_allocated(),
            'cuda_allocated_human': human_bytes(torch.cuda.memory_allocated()),
            'cuda_reserved_human': human_bytes(torch.cuda.memory_reserved()),
            'cuda_max_allocated_human': human_bytes(torch.cuda.max_memory_allocated()),
        })
    return item


def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    return text.strip().split('\n')[0].strip()


def load_runtime_model(model_dir):
    runtime_dir = Path(model_dir) / 'runtime'
    runtime_str = str(runtime_dir)
    if runtime_str in sys.path:
        sys.path.remove(runtime_str)
    sys.path.insert(0, runtime_str)
    for module_name in ['triton_loader', 'native_loader', 'loader']:
        if module_name in sys.modules:
            del sys.modules[module_name]

    import importlib
    triton_loader = importlib.import_module('triton_loader')
    load_fn = getattr(triton_loader, 'load_triton_packed_recovered_l78')
    t0 = time.perf_counter()
    model, tokenizer, report = load_fn(str(model_dir), token=os.environ.get('HF_TOKEN'))
    load_elapsed = time.perf_counter() - t0
    report = dict(report)
    report['selected_loader'] = 'triton_loader.load_triton_packed_recovered_l78'
    model.eval()
    return model, tokenizer, report, load_elapsed


def run_generation(model, tokenizer, question, max_new_tokens=MAX_NEW_TOKENS):
    prompt = PROMPT_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = extract_answer(prompt, generated)
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    total_tokens = int(out.shape[-1])
    generated_tokens = max(0, total_tokens - prompt_tokens)
    return {
        'question': question,
        'prompt_tokens': prompt_tokens,
        'generated_tokens': generated_tokens,
        'answer': answer,
        'elapsed_sec': elapsed,
        'tokens_per_sec': (generated_tokens / elapsed) if elapsed > 0 and generated_tokens > 0 else None,
    }


In [ ]:
# Reconstrói distribuível e monta bundle de produção.
runtime_downloaded = ensure_distributable()

if PROD_DIR.exists() or PROD_DIR.is_symlink():
    shutil.rmtree(PROD_DIR)
PROD_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DIST_DIR, PROD_DIR)

# Metadados adicionais de produção.
production_readme = PROD_DIR / 'PRODUCTION_USAGE.txt'
production_readme.write_text(
    '\n'.join([
        'RPT production Triton bundle',
        '',
        'Linux/NVIDIA/Colab usage:',
        'python runtime/run_triton_inference.py --model-dir . --prompt "Qual a capital da França?"',
        '',
        'Requirements:',
        '- CUDA-compatible NVIDIA GPU',
        '- Python environment with requirements.txt installed',
        '- HF_TOKEN if the base/tokenizer path needs authenticated access',
        '',
        'This package was prewarmed for common prompt shapes but first-run latency can still vary by hardware/runtime cache state.',
    ]) + '\n',
    encoding='utf-8'
)
print('production bundle ready:', PROD_DIR)
print('runtime_downloaded =', runtime_downloaded)


In [ ]:
# Prewarm do bundle de produção.
cleanup_memory()
memory_before = snapshot_memory('before_load')
model, tokenizer, runtime_report, load_elapsed = load_runtime_model(PROD_DIR)
memory_after = snapshot_memory('after_load')

prewarm_rows = []
for question in PREWARM_QUESTIONS:
    row = run_generation(model, tokenizer, question)
    row['phase'] = 'prewarm'
    prewarm_rows.append(row)

verification_rows = []
for question in PREWARM_QUESTIONS:
    row = run_generation(model, tokenizer, question)
    row['phase'] = 'verification_after_prewarm'
    verification_rows.append(row)

prewarm_profile = {
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model_dir': str(PROD_DIR),
    'load_elapsed_sec': load_elapsed,
    'runtime_report': runtime_report,
    'memory_before_load': memory_before,
    'memory_after_load': memory_after,
    'prewarm_rows': prewarm_rows,
    'verification_rows': verification_rows,
    'verification_summary': {
        'avg_tokens_per_sec': statistics.mean([r['tokens_per_sec'] for r in verification_rows if r['tokens_per_sec'] is not None]),
        'max_latency_sec': max(r['elapsed_sec'] for r in verification_rows),
        'min_latency_sec': min(r['elapsed_sec'] for r in verification_rows),
    },
}
(PROD_DIR / 'prewarm_profile.json').write_text(json.dumps(prewarm_profile, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(prewarm_profile['verification_summary'], ensure_ascii=False, indent=2))


In [ ]:
# Manifesto de produção e zip.
package_manifest_path = PROD_DIR / 'runtime' / 'package_manifest.template.json'
base_manifest = json.loads(package_manifest_path.read_text(encoding='utf-8')) if package_manifest_path.exists() else {}
production_manifest = {
    **base_manifest,
    'name': PROD_PACKAGE_NAME,
    'version': '0.2.0',
    'runtime_kind': 'native_triton_prototype',
    'is_native_ternary_runtime': True,
    'is_optimized_kernel': True,
    'load_order': 'student_pruned -> recovered adapter -> replace MLP base layers with TritonPackedTernaryLinear',
    'expected_packed_sha256': runtime_report.get('packed_sha256'),
    'expected_packed_mlp_tensors': runtime_report.get('replaced_count', 78),
    'validated_local_accuracy': {'benchmark': 'gemma4_100_questions', 'correct': 100, 'total': 100, 'accuracy': 1.0},
    'post_prewarm_runtime_summary': json.loads((PROD_DIR / 'prewarm_profile.json').read_text(encoding='utf-8'))['verification_summary'],
    'target_environment': ['linux', 'nvidia', 'colab'],
    'entrypoint': 'runtime/run_triton_inference.py',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
(PROD_DIR / 'production_manifest.json').write_text(json.dumps(production_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

report = {
    'notebook': '27_build_production_triton_bundle',
    'purpose': 'Build downloadable production-like Triton runtime bundle for Linux/NVIDIA/Colab.',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'production_dir': str(PROD_DIR),
    'runtime_report': runtime_report,
    'runtime_downloaded': runtime_downloaded,
    'load_elapsed_sec': load_elapsed,
    'memory_before_load': memory_before,
    'memory_after_load': memory_after,
    'prewarm_profile_path': str(PROD_DIR / 'prewarm_profile.json'),
    'production_manifest_path': str(PROD_DIR / 'production_manifest.json'),
}
report_path = PROD_RESULTS_DIR / f"27_build_production_triton_bundle_{time.strftime('%Y%m%d_%H%M%S')}.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

zip_base = PROD_RESULTS_DIR / PROD_PACKAGE_NAME
if (zip_base.with_suffix('.zip')).exists():
    (zip_base.with_suffix('.zip')).unlink()
shutil.make_archive(str(zip_base), 'zip', root_dir=str(PROD_DIR.parent), base_dir=PROD_DIR.name)
zip_path = zip_base.with_suffix('.zip')
print('report:', report_path)
print('zip:', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
